In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import keras
import tensorflow as tf
from sklearn.model_selection import train_test_split

In [ ]:
df_train = pd.read_csv("dataset/spaceship-titanic/train.csv")
df_test = pd.read_csv("dataset/spaceship-titanic/test.csv")
print(df_train.head())
print(df_train.info())
print(df_train.shape)

In [ ]:
# null 값 확인
def count_nulls(df):
    null_counts = df.isnull().sum()
    return null_counts


before_preprocess = count_nulls(df_train)
print(before_preprocess)

In [ ]:
# null값이 꽤 분포함. 아직 MCAR인지 MAR인지 MNAR인지 판단하기 어려움.

# todo:
# 1. passenger id는 가족 가능성 있음. family id로 만들어보자.
#    만약 family id가 있다면, CryoSleep, VIP, Cabin, Destination, RoomService, FoodCourt, ShoppingMall, Spa, VRDeck 연관이 있을 수 있음.
#    만약 Name에서 Mr, Mrs, Miss 등으로 나눌 수 있다면, 성별과 연관이 있을 수 있음.
#    만약 family name이 있다면, 이름에서 family name을 추출해서 family id로 만들어보자.
# 2. cabin은 deck, num, side로 나눠보자.
# 3. age는 범주형으로 바꿔보자.

In [ ]:
# passenger id는 가족 가능성 있음. family id로 만들어보자.
def extract_family_id(passenger_id):
    return passenger_id.split("_")[0]


def extract_family_member_id(passenger_id):
    return passenger_id.split("_")[1]


def transform_passenger_id(df):
    df["FamilyID"] = df["PassengerId"].apply(extract_family_id).astype("int")
    df["FamilyMemberID"] = (
        df["PassengerId"].apply(extract_family_member_id).astype("int")
    )
    return df


def extract_deck(cabin):
    return cabin.split("/")[0] if pd.notnull(cabin) else np.nan


def extract_num(cabin):
    return cabin.split("/")[1] if pd.notnull(cabin) else np.nan


def extract_side(cabin):
    return cabin.split("/")[2] if pd.notnull(cabin) else np.nan


def transform_cabin(df):
    df["Deck"] = df["Cabin"].apply(extract_deck)
    df["Num"] = df["Cabin"].apply(extract_num).astype("float")
    df["Side"] = df["Cabin"].apply(extract_side)
    return df


def extract_age_group(age):
    if pd.isnull(age):
        return np.nan
    elif age < 12:
        return "Child"
    elif age < 18:
        return "Teen"
    elif age < 30:
        return "Young Adult"
    elif age < 60:
        return "Adult"
    else:
        return "Senior"


def transform_age(df):
    df["AgeGroup"] = df["Age"].apply(extract_age_group)
    return df


# Home Planet과 Destination은 범주형 변수이므로, 이를 수치형으로 변환
planet_mapping = {"Earth": 0, "Europa": 1, "Mars": 2}
destination_mapping = {"TRAPPIST-1e": 0, "PSO J318.5-22": 1, "55 Cancri e": 2}


def transform_planet_and_destination(df):
    df["HomePlanet"] = df["HomePlanet"].map(planet_mapping)
    df["Destination"] = df["Destination"].map(destination_mapping)
    return df


# true/false 값을 1/0으로 변환
def transform_boolean(df, column):
    # null 값이 있는 경우, np.nan로 처리
    df[column] = df[column].map({True: 1, False: 0, np.nan: np.nan})
    return df


# name에서 family name 추출하기
def extract_family_name(name):
    if pd.isnull(name):
        return ""
    if " " not in name:
        return ""
    family_name = name.split(" ")[-1].strip()
    return family_name


def transform_name(df):
    df["FamilyName"] = df["Name"].apply(extract_family_name)
    df["HasSameFamilyNameInFamilyGroup"] = df.groupby("FamilyID")[
        "FamilyName"
    ].transform(lambda x: x.duplicated(keep=False).astype(int))
    return df


rel_family_features = [
    "CryoSleep",
    "VIP",
    "Cabin",
    "HomePlanet",
    "Destination",
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck",
]

one_hot_features = ["Deck", "Side", "AgeGroup"]


def transform_missing_if_family_member_has_same_family_name(df):
    # family member가 같은 family name을 가지고 있다면, null 값을 탐색해서 family 값으로 채워보자. (카테고리 때문에 평균은 안됌)
    for family_id, group in df.groupby("FamilyID"):
        if group["HasSameFamilyNameInFamilyGroup"].iloc[0] == 1:
            for column in rel_family_features:
                if group[column].isnull().any():
                    # null 값이 있는 경우, null이 아닌 값으로 채워보자.
                    # cabin의 경우 apply로 deck, num, side로 나눠서 채워야 할 수도 있음.
                    if column == "Cabin":
                        for sub_column in ["Deck", "Num", "Side"]:
                            if group[sub_column].isnull().any():
                                try:
                                    non_null_value = group[sub_column].dropna().iloc[0]
                                    if sub_column == "Num":
                                        non_null_value = float(non_null_value)
                                    df.loc[group.index, sub_column] = df.loc[
                                        group.index, sub_column
                                    ].fillna(non_null_value)
                                except IndexError:
                                    continue
                    try:
                        non_null_value = group[column].dropna().iloc[0]
                        if column == "Num":
                            non_null_value = float(non_null_value)
                        df.loc[group.index, column] = df.loc[
                            group.index, column
                        ].fillna(non_null_value)
                    except IndexError:
                        continue
    return df


def preprocessing_data(df):
    df = transform_passenger_id(df)
    df = transform_cabin(df)
    df = transform_age(df)
    df = transform_planet_and_destination(df)
    df = transform_boolean(df, "VIP")
    df = transform_boolean(df, "CryoSleep")
    df = transform_name(df)
    df = transform_missing_if_family_member_has_same_family_name(df)
    df = pd.get_dummies(df, columns=one_hot_features, dummy_na=True)
    df.drop(columns=["Cabin", "Age", "Name", "FamilyName", "FamilyID"], inplace=True)
    return df


df_train = preprocessing_data(df_train)
print(df_train.head())

In [ ]:
after_preprocess = count_nulls(df_train)
# print(after_preprocess)

print(df_train.info())


def show_diff(before, after):
    diff = before - after
    print(diff)


# show_diff(before_preprocess, after_preprocess)

In [ ]:
train_data, val_data = train_test_split(df_train, test_size=0.2, random_state=42)
X_train = train_data.drop(columns=["Transported"])
y_train = train_data["Transported"]
X_val = val_data.drop(columns=["Transported"])
y_val = val_data["Transported"]

import xgboost as xgb

model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)
model.fit(X_train[[col for col in X_train.columns if col != "PassengerId"]], y_train)
y_pred = model.predict(X_val[[col for col in X_val.columns if col != "PassengerId"]])
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_val, y_pred)
print(f"Validation Accuracy: {accuracy:.4f}")

In [ ]:
# test 데이터도 동일하게 전처리
df_test = preprocessing_data(df_test)
X_test = df_test[[col for col in df_test.columns if col != "PassengerId"]]
test_predictions = model.predict(X_test)
# map to 0/1 -> True/False
test_predictions = np.where(test_predictions == 1, True, False)
submission = pd.DataFrame(
    {"PassengerId": df_test["PassengerId"], "Transported": test_predictions}
)
submission.to_csv("submission.csv", index=False)